In [ ]:
import pandas as pd
import numpy as np
import joblib
import yaml
import os
import json
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# Load Config
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

# Define targets to compare
TARGETS = ['DEATH', 'THIRTY_DAY_MORT', 'ONE_YEAR_MORT', 'FIVE_YEAR_MORT', 'LOG_LOS']

print(f"Targets to compare: {TARGETS}")

In [ ]:
def load_and_prepare_data(dataset_path, target_col):
    """
    Loads dataset, aggregates it to create weights, and returns X, y, and weights.
    """
    full_path = f"../datasets/{dataset_path}"
    if not os.path.exists(full_path):
        print(f"Error: File {full_path} not found.")
        return None, None, None
        
    df = pd.read_csv(full_path)
    
    # Aggregate to handle duplicates and create sample weights 'N'
    # This matches the training process
    df_agg = df.groupby(list(df.columns), dropna=False).size().reset_index(name='N')
    
    # Identify input columns (C_...)
    input_cols = [c for c in df.columns if c.startswith("C_")]
    
    X = df_agg[input_cols]
    y = df_agg[target_col]
    weights = df_agg['N']
    
    return X, y, weights

In [ ]:
models_dir = "../models"
weights_path = "../original_weights.json"

# Load original weights if available
if os.path.exists(weights_path):
    with open(weights_path, "r") as f:
        original_weights = json.load(f)
else:
    print(f"Warning: {weights_path} not found. Original weight evaluation will be skipped.")
    original_weights = None

for target_col in TARGETS:
    print(f"\n{'='*50}")
    print(f"Processing Target: {target_col}")
    print(f"{'='*50}")
    
    if target_col == 'LOG_LOS':
        task_type = 'los'
        metric_name = 'R2 Score'
        is_classification = False
    else:
        task_type = 'mort'
        metric_name = 'AUC'
        is_classification = True
        
    results = []
    
    # --- Evaluate Trained Models ---
    methods = ['elixhauser', 'charlson', 'macss']
    
    for method in methods:
        model_filename = f"{task_type}_{method}_{target_col}.joblib"
        model_path = os.path.join(models_dir, model_filename)
        
        if os.path.exists(model_path):
            print(f"Evaluating {method.capitalize()} (Trained)...")
            try:
                model = joblib.load(model_path)
                
                # Determine dataset key from config
                # We need to find a config entry that matches this method and target class
                # to get the correct testing dataset file.
                # look for keys starting with method and check 'class'
                dataset_filename = None
                for key, conf in config['models'].items():
                    if key.startswith(method) and conf.get('class') == target_col:
                        dataset_filename = conf['dataset_testing']
                        break
                
                if dataset_filename:
                    X, y, w = load_and_prepare_data(dataset_filename, target_col)
                    
                    if X is not None:
                        if is_classification:
                            y_pred = model.predict_proba(X)[:, 1]
                            score = roc_auc_score(y, y_pred, sample_weight=w)
                        else:
                            y_pred = model.predict(X)
                            score = r2_score(y, y_pred, sample_weight=w)
                        
                        results.append({"Method": f"{method.capitalize()} (Trained)", metric_name: score})
                else:
                    print(f"  Could not find config entry for {method} and {target_col}")
                    
            except Exception as e:
                print(f"  Error evaluating {method} (Trained): {e}")
        else:
            print(f"  Model not found: {model_filename}")

    # --- Evaluate Original Weights ---
    if original_weights:
        for method in ['elixhauser', 'charlson']:
            # Find dataset again
            dataset_filename = None
            for key, conf in config['models'].items():
                if key.startswith(method) and conf.get('class') == target_col:
                    dataset_filename = conf['dataset_testing']
                    break
            
            if dataset_filename:
                dataset_path = f"../datasets/{dataset_filename}"
                if os.path.exists(dataset_path):
                    print(f"Evaluating {method.capitalize()} (Original Weights)...")
                    try:
                        df = pd.read_csv(dataset_path)
                        
                        # Calculate score
                        weights = original_weights.get(method, {})
                        valid_weights = {k: v for k, v in weights.items() if k in df.columns}
                        
                        if valid_weights:
                            df['score'] = df[list(valid_weights.keys())].mul(pd.Series(valid_weights)).sum(axis=1)
                        else:
                            df['score'] = 0
                            
                        if is_classification:
                            score = roc_auc_score(df[target_col], df['score'])
                        else:
                            lr = LinearRegression()
                            mask = df[target_col].notna() & df['score'].notna()
                            if mask.sum() > 0:
                                X_score = df.loc[mask, ['score']]
                                y_true = df.loc[mask, target_col]
                                lr.fit(X_score, y_true)
                                score = lr.score(X_score, y_true)
                            else:
                                score = np.nan
                                
                        results.append({"Method": f"{method.capitalize()} (Original Weights)", metric_name: score})
                    except Exception as e:
                         print(f"  Error evaluating {method} (Original Weights): {e}")
                else:
                    print(f"  Dataset not found: {dataset_path}")

    # --- Display Results for this Target ---
    if results:
        results_df = pd.DataFrame(results).sort_values(by=metric_name, ascending=False)
        print(f"\nResults for {target_col}:")
        print(results_df)

        plt.figure(figsize=(10, 6))
        bars = plt.barh(results_df["Method"], results_df[metric_name], color='skyblue')
        plt.xlabel(metric_name)
        plt.title(f"Model Performance Comparison - {target_col}")
        
        if is_classification:
            plt.xlim(0.5, 1.0)
        else:
             # Dynamic limit for R2
             max_val = results_df[metric_name].max()
             plt.xlim(0, max(max_val * 1.1, 0.1) if not pd.isna(max_val) else 1.0)

        plt.grid(axis='x', linestyle='--', alpha=0.7)
        
        for bar in bars:
            width = bar.get_width()
            plt.text(width, bar.get_y() + bar.get_height()/2, f'{width:.4f}', 
                     ha='left', va='center', fontweight='bold')

        plt.tight_layout()
        plt.show()
    else:
        print(f"No results generated for {target_col}")